# Azure-Deployed Interactive Brokers HFT – Visualization

This notebook generates all figures used in the project README and saves them under `docs/figures/`.

Run all cells top-to-bottom to regenerate the visual assets.

In [ ]:
import os

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
import seaborn as sns
import networkx as nx
import numpy as np

FIG_DIR = "docs/figures"
os.makedirs(FIG_DIR, exist_ok=True)

# Global style configuration
mpl.rcParams.update({
    "figure.figsize": (12, 7),
    "axes.facecolor": "#0f172a",  # slate-900
    "figure.facecolor": "#020617",  # slate-950
    "axes.edgecolor": "#e5e7eb",   # gray-200
    "axes.labelcolor": "#e5e7eb",
    "xtick.color": "#e5e7eb",
    "ytick.color": "#e5e7eb",
    "text.color": "#e5e7eb",
    "grid.color": "#1f2937",
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.4,
    "font.size": 11,
    "legend.frameon": False,
})

PALETTE = {
    "primary": "#0ea5e9",    # sky-500
    "secondary": "#22c55e",  # emerald-500
    "accent": "#f97316",     # orange-500
    "muted": "#64748b",      # slate-500
    "critical": "#ef4444",   # red-500
}

sns.set_theme(style="darkgrid")
sns.set_palette([PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"], PALETTE["muted"]])

def save_figure(fig, filename: str):
    path = os.path.join(FIG_DIR, filename)
    fig.savefig(path, dpi=220, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    return path

FIG_DIR

## Figure 1 – System Architecture Diagram

IB Gateway → Market Data Collector → Redis Streams → TimescaleDB, Strategy Engine → Risk/OMS → Kafka/Event Hubs → Order Execution, wrapped in Azure AKS with peripheral managed services.

In [ ]:
def generate_figure_01_system_architecture():
    fig, ax = plt.subplots(figsize=(14, 8))
    fig.patch.set_facecolor(mpl.rcParams["figure.facecolor"])
    ax.set_facecolor(mpl.rcParams["axes.facecolor"])
    ax.set_axis_off()

    # High-level AKS boundary
    aks_box = patches.FancyBboxPatch(
        (0.05, 0.15), 0.9, 0.7,
        boxstyle="round,pad=0.02,rounding_size=10",
        edgecolor=PALETTE["primary"],
        linestyle="--",
        linewidth=1.5,
        facecolor=(0, 0, 0, 0.02),
    )
    ax.add_patch(aks_box)
    ax.text(0.5, 0.84, "Azure Kubernetes Service (AKS)",
            ha="center", va="center", fontsize=14, color=PALETTE["primary"], fontweight="bold")

    # Component positions (normalized axes coordinates)
    components = {
        "IB Gateway": (0.08, 0.6),
        "Market Data\nCollector": (0.25, 0.6),
        "Redis Streams": (0.42, 0.6),
        "TimescaleDB\n(Postgres)": (0.59, 0.6),
        "Strategy Engine": (0.25, 0.4),
        "Risk / OMS": (0.42, 0.4),
        "Kafka /\nEvent Hubs": (0.59, 0.4),
        "Order Execution": (0.76, 0.4),
    }

    def draw_node(label, xy, color=PALETTE["secondary"], width=0.12, height=0.12):
        x, y = xy
        box = patches.FancyBboxPatch(
            (x - width / 2, y - height / 2), width, height,
            boxstyle="round,pad=0.03,rounding_size=6",
            edgecolor=color,
            facecolor=(0.02, 0.12, 0.18, 0.95),
            linewidth=1.4,
        )
        ax.add_patch(box)
        ax.text(x, y, label, ha="center", va="center", fontsize=10)

    # Draw main components
    for name, pos in components.items():
        draw_node(name, pos)

    # Peripheral services
    peripherals = {
        "Azure Key Vault": (0.82, 0.7),
        "Azure Event Hubs": (0.82, 0.6),
        "Application\nInsights": (0.82, 0.5),
    }
    for name, pos in peripherals.items():
        draw_node(name, pos, color=PALETTE["accent"], width=0.14, height=0.1)

    # Directed arrows between main path components
    arrow_kwargs = dict(
        arrowstyle="-|>",
        mutation_scale=16,
        color=PALETTE["muted"],
        linewidth=1.2,
    )

    def connect(src, dst):
        x1, y1 = components[src]
        x2, y2 = components[dst]
        ax.annotate("", xy=(x2 - 0.06, y2), xytext=(x1 + 0.06, y1), arrowprops=arrow_kwargs)

    connect("IB Gateway", "Market Data\nCollector")
    connect("Market Data\nCollector", "Redis Streams")
    connect("Redis Streams", "TimescaleDB\n(Postgres)")
    connect("Strategy Engine", "Risk / OMS")
    connect("Risk / OMS", "Kafka /\nEvent Hubs")
    connect("Kafka /\nEvent Hubs", "Order Execution")

    # Links between data and strategy path
    ax.annotate("", xy=(components["Strategy Engine"][0], components["Strategy Engine"][1] + 0.08),
                xytext=(components["Redis Streams"][0], components["Redis Streams"][1] - 0.08),
                arrowprops=arrow_kwargs)

    # Peripheral arrows
    ax.annotate("secrets", xy=peripherals["Azure Key Vault"],
                xytext=(components["Risk / OMS"][0] + 0.02, components["Risk / OMS"][1] + 0.08),
                arrowprops=arrow_kwargs, ha="center", fontsize=8)
    ax.annotate("orders / events", xy=peripherals["Azure Event Hubs"],
                xytext=(components["Kafka /\nEvent Hubs"][0] + 0.02, components["Kafka /\nEvent Hubs"][1] + 0.08),
                arrowprops=arrow_kwargs, ha="center", fontsize=8)
    ax.annotate("metrics / traces", xy=peripherals["Application\nInsights"],
                xytext=(components["Strategy Engine"][0] + 0.02, components["Strategy Engine"][1] - 0.08),
                arrowprops=arrow_kwargs, ha="center", fontsize=8)

    ax.set_title("System Architecture – Azure-Deployed Interactive Brokers HFT", fontsize=14, pad=16)

    return save_figure(fig, "figure_01_system_architecture.png")


generate_figure_01_system_architecture()

## Figure 2 – Latency Waterfall Chart

Latency budget for each critical stage of the trading pipeline.

In [ ]:
def generate_figure_02_latency_waterfall():
    stages = [
        "Data Ingestion",
        "Risk Checks",
        "Order Processing",
        "Signal Generation",
    ]
    upper_bounds = [5, 10, 50, 100]
    colors = [
        PALETTE["primary"],
        PALETTE["secondary"],
        PALETTE["critical"],
        PALETTE["accent"],
    ]

    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor(mpl.rcParams["figure.facecolor"])
    ax.set_facecolor(mpl.rcParams["axes.facecolor"])

    y_pos = np.arange(len(stages))
    bars = ax.barh(y_pos, upper_bounds, color=colors, alpha=0.9)

    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax.text(width + max(upper_bounds) * 0.02, bar.get_y() + bar.get_height() / 2,
                f"< {upper_bounds[i]} ms", va="center", fontsize=10)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(stages)
    ax.set_xlabel("Latency Budget (milliseconds)")
    ax.set_title("Latency Waterfall – End-to-End Trading Pipeline", pad=16)

    # Criticality annotation
    ax.text(0.01, 0.05,
            "Tight budgets for ingestion and risk checks enable sub-100 ms signal-to-order latency.",
            transform=ax.transAxes, fontsize=9, color=PALETTE["muted"])

    ax.set_xlim(0, max(upper_bounds) * 1.25)

    return save_figure(fig, "figure_02_latency_waterfall.png")


generate_figure_02_latency_waterfall()

## Figure 3 – Throughput Dashboard

Multi-panel view of key throughput metrics across the system.

In [ ]:
def gauge_axes(ax, label, value, max_value, unit, color):
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    # Draw arc
    theta = np.linspace(-np.pi/2, np.pi/2, 200)
    ax.plot(theta, np.ones_like(theta), color=PALETTE["muted"], linewidth=6, alpha=0.4)

    # Needle
    angle = -np.pi/2 + (value / max_value) * np.pi
    ax.plot([angle, angle], [0, 1], color=color, linewidth=4)

    ax.text(0.5, -0.1, label, transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(0.5, 0.15, f"{value:,.0f} {unit}", transform=ax.transAxes,
            ha="center", va="center", fontsize=11, fontweight="bold", color=color)


def generate_figure_03_throughput_metrics():
    fig = plt.figure(figsize=(14, 7))
    fig.patch.set_facecolor(mpl.rcParams["figure.facecolor"])
    gs = gridspec.GridSpec(2, 3, height_ratios=[1, 1.1])

    # Gauges
    ax1 = plt.subplot(gs[0, 0], projection="polar")
    ax2 = plt.subplot(gs[0, 1], projection="polar")
    ax3 = plt.subplot(gs[0, 2], projection="polar")

    gauge_axes(ax1, "Market Data", 10000, 15000, "ticks / sec", PALETTE["primary"])
    gauge_axes(ax2, "Order Processing", 1000, 2000, "orders / sec", PALETTE["secondary"])
    gauge_axes(ax3, "Signal Generation", 100, 300, "signals / sec", PALETTE["accent"])

    # Bar chart for storage throughput
    ax4 = plt.subplot(gs[1, :])
    ax4.set_facecolor(mpl.rcParams["axes.facecolor"])

    categories = ["Market Data", "Orders", "Signals", "Time-Series Storage"]
    values = [10000, 1000, 100, 1_000_000 / 60]  # points per second
    colors = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"], "#a855f7"]

    bars = ax4.bar(categories, values, color=colors, alpha=0.9)
    ax4.set_ylabel("Throughput (per second)")
    ax4.set_yscale("log")
    ax4.set_title("Throughput Metrics – Market Data, Orders, Signals, Storage", pad=10)

    for bar, val in zip(bars, values):
        ax4.text(bar.get_x() + bar.get_width() / 2, val * 1.2, f"{val:,.0f}",
                 ha="center", va="bottom", fontsize=9)

    ax4.text(0.01, -0.28,
             "Dashboard-style view of sustained throughput across critical system components.",
             transform=ax4.transAxes, fontsize=9, color=PALETTE["muted"])

    fig.suptitle("Throughput Dashboard – High-Frequency Trading System", fontsize=14, y=0.98)

    return save_figure(fig, "figure_03_throughput_metrics.png")


generate_figure_03_throughput_metrics()

## Figure 4 – Deployment Pipeline Flow

Backtest → Paper Trading → Shadow-Live → Production, with validation gates and tooling overlays (Terraform, Docker, Kubernetes).

In [ ]:
def generate_figure_04_deployment_pipeline():
    fig, ax = plt.subplots(figsize=(13, 4))
    fig.patch.set_facecolor(mpl.rcParams["figure.facecolor"])
    ax.set_facecolor(mpl.rcParams["axes.facecolor"])
    ax.set_axis_off()

    stages = ["Backtest", "Paper Trading", "Shadow-Live", "Production"]
    x_positions = [0.12, 0.35, 0.6, 0.85]

    def draw_stage(label, x, color):
        box = patches.FancyBboxPatch(
            (x - 0.1, 0.45), 0.2, 0.22,
            boxstyle="round,pad=0.03,rounding_size=6",
            edgecolor=color,
            facecolor=(0.02, 0.12, 0.18, 0.95),
            linewidth=1.5,
        )
        ax.add_patch(box)
        ax.text(x, 0.56, label, ha="center", va="center", fontsize=11)

    colors = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"], PALETTE["critical"]]
    for s, x, c in zip(stages, x_positions, colors):
        draw_stage(s, x, c)

    # Validation gates
    for xm in [(x_positions[i] + x_positions[i+1]) / 2 for i in range(len(x_positions) - 1)]:
        gate = patches.FancyBboxPatch(
            (xm - 0.03, 0.48), 0.06, 0.16,
            boxstyle="round,pad=0.01,rounding_size=4",
            edgecolor=PALETTE["muted"],
            facecolor=(0.03, 0.07, 0.12, 0.95),
            linewidth=1.0,
        )
        ax.add_patch(gate)
        ax.text(xm, 0.56, "QA\nGate", ha="center", va="center", fontsize=8, color=PALETTE["muted"])

    # Arrows
    arrow_kwargs = dict(arrowstyle="-|>", mutation_scale=16, color=PALETTE["muted"], linewidth=1.3)
    for i in range(len(x_positions) - 1):
        ax.annotate("", xy=(x_positions[i+1] - 0.12, 0.56), xytext=(x_positions[i] + 0.12, 0.56), arrowprops=arrow_kwargs)

    # Tooling overlays
    ax.text(0.12, 0.25, "Backtests, sim fills,\nPython + FastAPI", ha="center", va="center", fontsize=9, color=PALETTE["muted"])
    ax.text(0.35, 0.25, "Docker images, CI tests", ha="center", va="center", fontsize=9, color=PALETTE["muted"])
    ax.text(0.6, 0.25, "AKS manifests, Helm,\nKubernetes rollouts", ha="center", va="center", fontsize=9, color=PALETTE["muted"])
    ax.text(0.85, 0.25, "Terraform, Azure infra,\nblue/green deploys", ha="center", va="center", fontsize=9, color=PALETTE["muted"])

    ax.set_title("Deployment Pipeline – Backtest to Production", fontsize=14, pad=16)

    return save_figure(fig, "figure_04_deployment_pipeline.png")


generate_figure_04_deployment_pipeline()

## Figure 5 – Risk Management Framework

Layered controls: Pre-Trade Risk → Real-Time Monitoring → Emergency Controls, with example checks per layer.

In [ ]:
def generate_figure_05_risk_framework():
    fig, ax = plt.subplots(figsize=(7, 7))
    fig.patch.set_facecolor(mpl.rcParams["figure.facecolor"])
    ax.set_facecolor(mpl.rcParams["axes.facecolor"])
    ax.set_aspect('equal')
    ax.axis('off')

    center = (0.5, 0.5)
    radii = [0.18, 0.32, 0.46]
    colors = [PALETTE["secondary"], PALETTE["accent"], PALETTE["critical"]]
    labels = ["Pre-Trade Risk", "Real-Time Monitoring", "Emergency Controls"]
    details = [
        "Position & exposure limits\nMax order size\nRate limiting",
        "PnL drift alerts\nVolatility regimes\nLatency anomalies",
        "Global kill switch\nCircuit breakers\nEmergency de-risking",
    ]

    for r, c, label, text in zip(radii[::-1], colors[::-1], labels[::-1], details[::-1]):
        circle = patches.Circle(center, r, edgecolor=c, facecolor=(0.02, 0.12, 0.18, 0.95), linewidth=1.5)
        ax.add_patch(circle)

    for r, label in zip(radii, labels):
        ax.text(center[0], center[1] + r - 0.06, label,
                ha="center", va="center", fontsize=10, fontweight="bold")

    # Detail text bands
    y_offsets = [0.0, -0.22, -0.38]
    for offset, text in zip(y_offsets, details):
        ax.text(center[0], center[1] + offset, text, ha="center", va="center", fontsize=9, color=PALETTE["muted"])

    ax.set_title("Risk Management Framework – Defense in Depth", fontsize=14, pad=16)

    return save_figure(fig, "figure_05_risk_framework.png")


generate_figure_05_risk_framework()

## Figure 6 – Azure Infrastructure Map

High-level map of Azure services: AKS, PostgreSQL/TimescaleDB, Redis Cache, Event Hubs, Key Vault, Application Insights, and network boundaries.

In [ ]:
def generate_figure_06_azure_infrastructure():
    fig, ax = plt.subplots(figsize=(13, 7))
    fig.patch.set_facecolor(mpl.rcParams["figure.facecolor"])
    ax.set_facecolor(mpl.rcParams["axes.facecolor"])
    ax.axis('off')

    # Virtual Network boundary
    vnet_box = patches.FancyBboxPatch(
        (0.05, 0.12), 0.9, 0.75,
        boxstyle="round,pad=0.02,rounding_size=10",
        edgecolor=PALETTE["primary"],
        linestyle="--",
        linewidth=1.5,
        facecolor=(0.01, 0.06, 0.12, 0.8),
    )
    ax.add_patch(vnet_box)
    ax.text(0.08, 0.86, "Azure Virtual Network (VNet)", fontsize=12, color=PALETTE["primary"], fontweight="bold")

    # AKS cluster box
    aks_box = patches.FancyBboxPatch(
        (0.08, 0.22), 0.44, 0.6,
        boxstyle="round,pad=0.02,rounding_size=8",
        edgecolor=PALETTE["secondary"],
        linewidth=1.4,
        facecolor=(0.02, 0.11, 0.18, 0.9),
    )
    ax.add_patch(aks_box)
    ax.text(0.3, 0.78, "AKS Cluster", ha="center", va="center", fontsize=12, color=PALETTE["secondary"], fontweight="bold")

    # AKS workloads
    aks_components = {
        "Market Data\nServices": (0.18, 0.65),
        "Strategy Engine": (0.18, 0.5),
        "Risk / OMS": (0.18, 0.35),
        "API Gateway": (0.37, 0.65),
        "Monitoring\nSidecars": (0.37, 0.5),
    }

    for name, (x, y) in aks_components.items():
        box = patches.FancyBboxPatch(
            (x - 0.08, y - 0.06), 0.16, 0.12,
            boxstyle="round,pad=0.02,rounding_size=5",
            edgecolor=PALETTE["muted"],
            facecolor=(0.02, 0.14, 0.22, 0.95),
            linewidth=1.1,
        )
        ax.add_patch(box)
        ax.text(x, y, name, ha="center", va="center", fontsize=9)

    # Managed services
    services = {
        "PostgreSQL /\nTimescaleDB": (0.7, 0.65),
        "Azure Cache\nfor Redis": (0.7, 0.5),
        "Event Hubs /\nKafka": (0.7, 0.35),
        "Key Vault": (0.88, 0.65),
        "Application\nInsights": (0.88, 0.5),
    }

    for name, (x, y) in services.items():
        box = patches.FancyBboxPatch(
            (x - 0.08, y - 0.06), 0.16, 0.12,
            boxstyle="round,pad=0.02,rounding_size=5",
            edgecolor=PALETTE["accent"],
            facecolor=(0.04, 0.12, 0.20, 0.95),
            linewidth=1.2,
        )
        ax.add_patch(box)
        ax.text(x, y, name, ha="center", va="center", fontsize=9)

    # Private endpoints and NSG annotations
    ax.text(0.7, 0.25, "Private Endpoints\n+ NSG rules", ha="center", va="center", fontsize=9, color=PALETTE["muted"])

    arrow_kwargs = dict(arrowstyle="-|>", mutation_scale=15, color=PALETTE["muted"], linewidth=1.2)
    # Example data flows
    ax.annotate("", xy=(0.62, 0.65), xytext=(0.52, 0.65), arrowprops=arrow_kwargs)
    ax.annotate("", xy=(0.62, 0.5), xytext=(0.52, 0.5), arrowprops=arrow_kwargs)
    ax.annotate("", xy=(0.62, 0.35), xytext=(0.52, 0.35), arrowprops=arrow_kwargs)
    ax.annotate("", xy=(0.8, 0.65), xytext=(0.78, 0.65), arrowprops=arrow_kwargs)
    ax.annotate("", xy=(0.8, 0.5), xytext=(0.78, 0.5), arrowprops=arrow_kwargs)

    ax.set_title("Azure Infrastructure Map – Managed Services and Network Boundaries", fontsize=14, pad=16)

    return save_figure(fig, "figure_06_azure_infrastructure.png")


generate_figure_06_azure_infrastructure()

## Figure 7 – Data Flow Timeline

End-to-end lifecycle of a single trade from tick ingestion through P&L reconciliation.

In [ ]:
def generate_figure_07_data_flow():
    steps = [
        "Market tick arrives",
        "Cached in Redis",
        "Signal generated",
        "Risk check passed",
        "Order submitted",
        "Fill received",
        "Position updated",
        "P&L reconciled",
    ]

    # Hypothetical timestamps in ms relative to tick arrival
    times = [0, 1, 12, 18, 30, 45, 55, 70]

    fig, ax = plt.subplots(figsize=(14, 3.5))
    fig.patch.set_facecolor(mpl.rcParams["figure.facecolor"])
    ax.set_facecolor(mpl.rcParams["axes.facecolor"])

    ax.plot(times, [1]*len(times), marker="o", color=PALETTE["primary"], linewidth=2)
    ax.set_yticks([])
    ax.set_xlabel("Time since tick arrival (ms)")
    ax.set_ylim(0.8, 1.2)

    for t, step in zip(times, steps):
        ax.text(t, 1.08, step, rotation=45, ha="right", va="bottom", fontsize=8)
        ax.text(t, 0.92, f"{t} ms", ha="center", va="top", fontsize=8, color=PALETTE["muted"])

    ax.set_title("Trade Lifecycle – Data Flow Timeline", fontsize=14, pad=16)
    ax.grid(axis="y", False)
    ax.grid(axis="x", linestyle="--", alpha=0.3)

    return save_figure(fig, "figure_07_data_flow.png")


generate_figure_07_data_flow()

## Figure 8 – Monitoring & Observability Stack

Prometheus + Grafana for metrics, OpenTelemetry + Application Insights for traces, plus alerting and escalation paths.

In [ ]:
def generate_figure_08_monitoring_stack():
    fig, ax = plt.subplots(figsize=(13, 6))
    fig.patch.set_facecolor(mpl.rcParams["figure.facecolor"])
    ax.set_facecolor(mpl.rcParams["axes.facecolor"])
    ax.axis('off')

    # Stack layers
    layers = [
        ("Instrumentation", "OpenTelemetry SDKs\nCustom app metrics", 0.75, PALETTE["primary"]),
        ("Collection", "Prometheus scraping\nOTel collectors", 0.55, PALETTE["secondary"]),
        ("Storage", "Time-series DBs\nApplication Insights", 0.35, PALETTE["accent"]),
        ("Visualization & Alerts", "Grafana dashboards\nAlert rules & escalation", 0.15, PALETTE["critical"]),
    ]

    for title, text, y, color in layers:
        box = patches.FancyBboxPatch(
            (0.08, y), 0.84, 0.16,
            boxstyle="round,pad=0.03,rounding_size=6",
            edgecolor=color,
            facecolor=(0.02, 0.12, 0.20, 0.95),
            linewidth=1.4,
        )
        ax.add_patch(box)
        ax.text(0.12, y + 0.08, title, ha="left", va="center", fontsize=11, fontweight="bold", color=color)
        ax.text(0.5, y + 0.08, text, ha="center", va="center", fontsize=9, color=PALETTE["muted"])

    # Escalation flow
    ax.annotate("", xy=(0.9, 0.24), xytext=(0.9, 0.33),
                arrowprops=dict(arrowstyle="-|>", mutation_scale=15, color=PALETTE["muted"], linewidth=1.2))
    ax.text(0.9, 0.21, "On-call SRE / Trader", ha="center", va="center", fontsize=9)

    ax.set_title("Monitoring & Observability Stack", fontsize=14, pad=16)

    return save_figure(fig, "figure_08_monitoring_stack.png")


generate_figure_08_monitoring_stack()

## Generate All Figures

Helper cell to regenerate the full figure set in one go.

In [ ]:
def generate_all_figures():
    paths = {}
    paths["figure_01_system_architecture.png"] = generate_figure_01_system_architecture()
    paths["figure_02_latency_waterfall.png"] = generate_figure_02_latency_waterfall()
    paths["figure_03_throughput_metrics.png"] = generate_figure_03_throughput_metrics()
    paths["figure_04_deployment_pipeline.png"] = generate_figure_04_deployment_pipeline()
    paths["figure_05_risk_framework.png"] = generate_figure_05_risk_framework()
    paths["figure_06_azure_infrastructure.png"] = generate_figure_06_azure_infrastructure()
    paths["figure_07_data_flow.png"] = generate_figure_07_data_flow()
    paths["figure_08_monitoring_stack.png"] = generate_figure_08_monitoring_stack()
    return paths


generate_all_figures()